# Polybag Detection — YOLO OBB Training

This notebook trains a **YOLO11 Oriented Bounding Box (OBB)** model to detect polybags in conveyor belt scenes.

**What you need before starting:**
1. The dataset `.zip` file link shared by your team
2. A Google account (for Colab GPU access)

**Steps:**
1. Enable GPU runtime (Runtime → Change runtime type → T4 GPU)
2. Run all cells top to bottom
3. Paste the dataset link when prompted

---
**7 classes:** pink · blue · yellow · grey · green · red · teal polybag

## 1. Check GPU and Install Dependencies

In [ ]:
# Verify that a GPU is available
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️  No GPU detected!')
    print('Go to Runtime → Change runtime type → select T4 GPU, then re-run.')
else:
    print(result.stdout)

In [ ]:
# Install the ultralytics library (includes YOLO)
!pip install ultralytics --quiet

import ultralytics
ultralytics.checks()  # prints system info and confirms GPU

## 2. Download the Dataset

Paste the **Google Drive shared link** to the dataset zip file below.

The link should look like:  
`https://drive.google.com/file/d/XXXXXXXXXXXXXXXX/view?usp=sharing`

In [ ]:
# ── Paste your shared Google Drive link here ──────────────────────────────────
GDRIVE_LINK = 'https://drive.google.com/file/d/YOUR_FILE_ID_HERE/view?usp=sharing'
# ─────────────────────────────────────────────────────────────────────────────

import re, os

# Extract the file ID from the link
match = re.search(r'/d/([a-zA-Z0-9_-]+)', GDRIVE_LINK)
if not match:
    raise ValueError('Could not parse the Google Drive link. Make sure you copied the full URL.')
file_id = match.group(1)
print(f'File ID: {file_id}')

# Download using gdown
!pip install gdown --quiet
!gdown {file_id} -O /content/dataset.zip
print('Download complete.')

In [ ]:
# Extract the zip file
!unzip -q /content/dataset.zip -d /content/

# Show what was extracted
!ls /content/

## 3. Set Up Dataset Configuration

We tell YOLO where the images and labels are, and what the class names are.

In [ ]:
import yaml
from pathlib import Path

DATA_ROOT = Path('/content')

# Locate train / val / test image folders
# (adjust these if your zip has a different folder structure)
TRAIN_IMAGES = DATA_ROOT / 'synth_dataset_mcmot' / 'images'
VAL_IMAGES   = DATA_ROOT / 'synth_dataset_val'   / 'images'
TEST_IMAGES  = DATA_ROOT / 'synth_dataset_test'  / 'images'

for p in [TRAIN_IMAGES, VAL_IMAGES, TEST_IMAGES]:
    count = len(list(p.glob('*.png')))
    status = '✅' if count > 0 else '❌ NOT FOUND'
    print(f'{status}  {p.name}: {count} images')

In [ ]:
# Write the data.yaml configuration file
data_config = {
    'path':  str(DATA_ROOT),
    'train': str(TRAIN_IMAGES.relative_to(DATA_ROOT)),
    'val':   str(VAL_IMAGES.relative_to(DATA_ROOT)),
    'test':  str(TEST_IMAGES.relative_to(DATA_ROOT)),
    'nc':    7,
    'names': [
        'pink_polybag', 'blue_polybag', 'yellow_polybag',
        'grey_polybag', 'green_polybag', 'red_polybag', 'teal_polybag'
    ]
}

yaml_path = DATA_ROOT / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print('data.yaml written:')
print(yaml_path.read_text())

## 4. Preview Some Training Images

A quick sanity check — let's look at a few images with their ground truth labels overlaid.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

CLASS_COLORS = [
    '#FF69B4',  # pink
    '#4169E1',  # blue
    '#FFD700',  # yellow
    '#808080',  # grey
    '#32CD32',  # green
    '#FF4500',  # red
    '#008B8B',  # teal
]
CLASS_NAMES = data_config['names']

sample_images = random.sample(sorted(TRAIN_IMAGES.glob('*.png')), 4)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, img_path in zip(axes.flat, sample_images):
    img = np.array(Image.open(img_path))
    label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    h, w = img.shape[:2]

    ax.imshow(img)
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            cls = int(parts[0])
            pts = np.array(parts[1:]).reshape(4, 2) * [w, h]
            poly = plt.Polygon(pts, fill=False,
                               edgecolor=CLASS_COLORS[cls], linewidth=2)
            ax.add_patch(poly)
            ax.text(pts[0, 0], pts[0, 1], CLASS_NAMES[cls],
                    color=CLASS_COLORS[cls], fontsize=7, fontweight='bold')

    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.suptitle('Sample Training Images with Ground Truth Labels', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Train the Model

We use **YOLO11n-OBB** (nano = smallest and fastest) as the starting point.  
Training takes roughly **30–60 minutes** on a Colab T4 GPU for 50 epochs.

| Parameter | Value | Meaning |
|---|---|---|
| `epochs` | 50 | How many times the model sees the full dataset |
| `imgsz` | 1280 | Image size during training (pixels) |
| `batch` | 8 | Images processed at once (limited by GPU memory) |
| `patience` | 20 | Stop early if no improvement for 20 epochs |

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-obb.pt')  # downloads automatically if not present

results = model.train(
    data    = str(yaml_path),
    epochs  = 50,
    imgsz   = 1280,
    batch   = 8,
    patience= 20,       # early stopping
    name    = 'polybag_obb',
    project = '/content/runs',
    exist_ok= True,
)

print('\nTraining complete!')
print(f'Best weights saved to: {results.save_dir}/weights/best.pt')

## 6. Visualise Training Results

YOLO automatically saves loss curves, precision/recall graphs, and a confusion matrix.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

run_dir = Path(results.save_dir)

# Show training curves and confusion matrix side by side
plots = {
    'Loss curves':        run_dir / 'results.png',
    'Confusion matrix':   run_dir / 'confusion_matrix_normalized.png',
    'Precision-Recall':   run_dir / 'PR_curve.png',
    'F1 curve':           run_dir / 'F1_curve.png',
}

available = {k: v for k, v in plots.items() if v.exists()}
fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 6))
if len(available) == 1:
    axes = [axes]

for ax, (title, path) in zip(axes, available.items()):
    ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Run Inference on Test Images

Let's see how the trained model performs on images it has never seen.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from ultralytics import YOLO

# Load the best weights from training
best_model = YOLO(str(run_dir / 'weights' / 'best.pt'))

# Pick 6 random test images
test_imgs = random.sample(sorted(TEST_IMAGES.glob('*.png')), 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, img_path in zip(axes.flat, test_imgs):
    preds = best_model.predict(str(img_path), conf=0.3, verbose=False)[0]
    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]

    ax.imshow(img)

    if preds.obb is not None and len(preds.obb) > 0:
        corners = preds.obb.xyxyxyxy.cpu().numpy()  # (N, 4, 2)
        confs   = preds.obb.conf.cpu().numpy()
        classes = preds.obb.cls.cpu().numpy().astype(int)

        for pts, conf, cls in zip(corners, confs, classes):
            color = CLASS_COLORS[cls]
            poly  = plt.Polygon(pts, fill=False, edgecolor=color, linewidth=2)
            ax.add_patch(poly)
            label = f'{CLASS_NAMES[cls]} {conf:.2f}'
            ax.text(pts[0, 0], pts[0, 1] - 5, label,
                    color=color, fontsize=7, fontweight='bold',
                    bbox=dict(facecolor='black', alpha=0.4, pad=1, edgecolor='none'))
    else:
        ax.set_xlabel('No detections', color='red')

    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

# Legend
patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=patches, loc='lower center', ncol=7,
           fontsize=10, framealpha=0.8)

plt.suptitle('Model Predictions on Test Images (conf ≥ 0.3)', fontsize=14)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## 8. Evaluate on the Full Test Set

This gives official metrics: mAP50 and mAP50-95.

In [ ]:
metrics = best_model.val(
    data   = str(yaml_path),
    split  = 'test',
    imgsz  = 1280,
    batch  = 8,
    verbose= True,
)

print(f'\nmAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 9. Download the Trained Weights

Save the best model weights to your Google Drive or download them directly.

In [ ]:
# Option A — download directly to your computer
from google.colab import files
files.download(str(run_dir / 'weights' / 'best.pt'))

In [ ]:
# Option B — copy to your Google Drive
from google.colab import drive
drive.mount('/gdrive', force_remount=False)

import shutil
dest = '/gdrive/MyDrive/polybag_best.pt'
shutil.copy(str(run_dir / 'weights' / 'best.pt'), dest)
print(f'Saved to {dest}')